In [2]:

# STEP 5 — FEATURE ENGINEERING
# ============================================================
# Load your EDA cleaned file (original column names intact)
import pandas as pd
import numpy as np

df = pd.read_csv("HR_Cleaned_For_EDA.csv")   # 50,000 rows, 26 columns
print("Loaded shape:", df.shape)

Loaded shape: (50000, 26)


In [3]:
# ============================================================
# 🆕 NEW FEATURE 1: Experience Category
# ============================================================

# "Fresher vs Senior"

df['experience_category'] = pd.cut(
    df['years_of_experience'],   # column to bin
    bins=[-1, 0, 2, 5],          # boundaries: -1→0, 0→2, 2→5
    labels=['Fresher',           # 0 years         → Fresher
            'Junior',            # 1 to 2 years    → Junior
            'Senior']            # 3 to 5 years    → Senior
)

In [4]:
# Verify it worked
print("\n✅ Feature 1 — Experience Category:")
print(df['experience_category'].value_counts())
print("Sample:", df[['years_of_experience',
                      'experience_category']].head(6))




✅ Feature 1 — Experience Category:
experience_category
Junior     23474
Fresher    14952
Senior     11574
Name: count, dtype: int64
Sample:    years_of_experience experience_category
0                    1              Junior
1                    0             Fresher
2                    1              Junior
3                    0             Fresher
4                    0             Fresher
5                    1              Junior


In [5]:
# ============================================================
# 🆕 NEW FEATURE 2: Academic Performance Band
# ============================================================

df['academic_band'] = pd.cut(
    df['degree_percentage'],          # column to bin
    bins=[0, 60, 75, 85, 100],        # boundary points
    labels=['Low',                    # 0–60%    → Low
            'Medium',                 # 60–75%   → Medium
            'High',                   # 75–85%   → High
            'Distinction']            # 85–100%  → Distinction
)

print("\n✅ Feature 2 — Academic Band:")
print(df['academic_band'].value_counts())



✅ Feature 2 — Academic Band:
academic_band
Medium         26846
High           19222
Distinction     2856
Low             1076
Name: count, dtype: int64


In [6]:
# ============================================================
# 🆕 NEW FEATURE 3: Skills Match Level
# ============================================================


df['skills_level'] = pd.cut(
    df['skills_match_percentage'],    # column to bin
    bins=[0, 65, 80, 100],            # boundary points
    labels=['Low',                    # 45–65%  → Low match
            'Medium',                 # 65–80%  → Medium match
            'High']                   # 80–100% → High match
)

print("\n✅ Feature 3 — Skills Level:")
print(df['skills_level'].value_counts())



✅ Feature 3 — Skills Level:
skills_level
Medium    23180
High      15440
Low       11380
Name: count, dtype: int64


In [7]:
# ============================================================
# 🆕 NEW FEATURE 4: Interview Performance Category
# ============================================================


df['interview_category'] = pd.cut(
    df['technical_score'],            # column to bin
    bins=[0, 60, 75, 100],            # boundary points
    labels=['Weak',                   # 40–60  → Weak
            'Average',                # 60–75  → Average
            'Strong']                 # 75–100 → Strong
)

print("\n✅ Feature 4 — Interview Category:")
print(df['interview_category'].value_counts())



✅ Feature 4 — Interview Category:
interview_category
Average    23416
Strong     14013
Weak       12571
Name: count, dtype: int64


In [8]:
# ============================================================
# 🆕 NEW FEATURE 5: Placement Probability Score
# ============================================================

df['placement_score'] = (
    df['technical_score']        * 0.5 +   # 50% weight
    df['skills_match_percentage']* 0.3 +   # 30% weight
    df['communication_score']    * 0.2     # 20% weight
)

# Round to 2 decimal places for cleanliness
df['placement_score'] = df['placement_score'].round(2)

print("\n✅ Feature 5 — Placement Score:")
print(df['placement_score'].describe().round(2))
print("Sample scores:", df['placement_score'].head(5).tolist())


✅ Feature 5 — Placement Score:
count    50000.00
mean        69.42
std          7.19
min         42.50
25%         64.50
50%         69.40
75%         74.34
max         95.85
Name: placement_score, dtype: float64
Sample scores: [65.87, 70.17, 74.76, 72.28, 73.13]


In [9]:
# ============================================================
# ✅ FINAL CHECK — See All New Features Together
# ============================================================
print("\n" + "="*55)
print("NEW FEATURES CREATED:")
print("="*55)

new_features = ['experience_category', 'academic_band',
                'skills_level', 'interview_category',
                'placement_score']

print(f"\nDataset shape BEFORE: (50000, 26)")
print(f"Dataset shape AFTER : {df.shape}")
print(f"New columns added   : {len(new_features)}")

# Show a sample row with all new features
print("\nSample candidate profile with new features:")
print(df[['years_of_experience', 'experience_category',
          'degree_percentage',   'academic_band',
          'skills_match_percentage', 'skills_level',
          'technical_score',     'interview_category',
          'placement_score',     'status']].head(5).to_string())


NEW FEATURES CREATED:

Dataset shape BEFORE: (50000, 26)
Dataset shape AFTER : (50000, 31)
New columns added   : 5

Sample candidate profile with new features:
   years_of_experience experience_category  degree_percentage academic_band  skills_match_percentage skills_level  technical_score interview_category  placement_score      status
0                    1              Junior          75.856526          High                79.548913       Medium        58.221909               Weak            65.87  Not Placed
1                    0             Fresher          73.093588        Medium                73.316134       Medium        71.927978            Average            70.17  Not Placed
2                    1              Junior          90.196460   Distinction                75.466980       Medium        72.445041            Average            74.76      Placed
3                    0             Fresher          75.586731          High                73.676449       Medium        78

In [10]:
# ============================================================
# 💾 SAVE FEATURE ENGINEERED DATA
# ============================================================

# STEP 1: Save EDA version (readable — with original column names)
df.to_csv("HR_Cleaned_For_EDA.csv", index=False)
print(" EDA file updated! Shape:", df.shape)

# STEP 2: Encode the 4 new categorical columns for ML
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

new_cat_cols = ['experience_category', 'academic_band',
                'skills_level', 'interview_category']

for col in new_cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f"  {col} encoded ")

# placement_score is already a number — skip encoding

# STEP 3: Save ML version (all numbers — ready for ML)
df.to_csv("HR_Cleaned_For_ML.csv", index=False)
print(" ML file updated! Shape:", df.shape)

# STEP 4: Quick verification
print("\n Final Check:")
print("Columns:", df.shape[1])       # should be 31
print("Rows:", df.shape[0])          # should be 50000
print("Missing values:", df.isnull().sum().sum())  # should be 0


 EDA file updated! Shape: (50000, 31)
  experience_category encoded 
  academic_band encoded 
  skills_level encoded 
  interview_category encoded 
 ML file updated! Shape: (50000, 31)

 Final Check:
Columns: 31
Rows: 50000
Missing values: 0


In [11]:
df

,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,...,notice_period_days,layoff_history,employment_gap_months,relocation_willingness,status,experience_category,academic_band,skills_level,interview_category,placement_score
0,27,Male,65.061656,83.842578,75.856526,Computer Science,58.221909,89.566305,64.474484,79.548913,...,15.0,No,18.0,Not Willing,Not Placed,1,1,2,2,65.87
1,24,Male,67.885626,64.973305,73.093588,Electronics,71.927978,54.591971,61.077306,73.316134,...,0.0,No,0.0,Not Willing,Not Placed,0,3,2,0,70.17
2,33,Female,73.892471,68.834121,90.196460,Information Technology,72.445041,58.587088,79.494739,75.466980,...,0.0,No,3.0,Not Willing,Placed,1,0,2,0,74.76
3,31,Male,74.145568,76.255126,75.586731,Mechanical,78.855676,61.022065,53.740386,73.676449,...,0.0,Yes,6.0,Willing,Not Placed,0,1,2,1,72.28
4,28,Male,60.475937,65.786336,80.801010,Information Technology,68.286776,65.713731,61.438314,88.994847,...,0.0,No,3.0,Willing,Not Placed,0,1,0,0,73.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,29,Female,76.382971,82.705920,81.790211,Electronics,71.955587,74.280744,65.613587,61.997596,...,0.0,No,6.0,Willing,Not Placed,1,1,1,0,67.70
49996,32,Female,80.029965,68.803782,69.178840,Mechanical,53.488972,51.796635,75.373337,77.767081,...,30.0,No,0.0,Willing,Not Placed,0,3,2,2,65.15
49997,28,Male,61.801411,69.771590,79.777937,Mechanical,60.530457,40.000000,54.851899,54.749371,...,0.0,No,3.0,Willing,Not Placed,0,1,1,0,57.66
49998,34,Male,71.811897,76.840708,86.809844,Others,67.611868,44.373600,69.899354,92.333175,...,60.0,No,0.0,Not Willing,Not Placed,0,0,0,0,75.49


In [12]:
df.columns

Index(['age_years', 'gender', 'ssc_percentage', 'hsc_percentage',
       'degree_percentage', 'degree_specialization', 'technical_score',
       'aptitude_score', 'communication_score', 'skills_match_percentage',
       'certifications_count', 'internship_experience', 'years_of_experience',
       'career_switch_willingness', 'relevant_experience', 'previous_ctc_lpa',
       'expected_ctc_lpa', 'company_tier', 'job_role_match',
       'competition_level', 'bond_requirement', 'notice_period_days',
       'layoff_history', 'employment_gap_months', 'relocation_willingness',
       'status', 'experience_category', 'academic_band', 'skills_level',
       'interview_category', 'placement_score'],
      dtype='object')

In [13]:
# ============================================================
# MERGE NEW FEATURES INTO MAIN ML FILE
# ============================================================

# Load your main cleaned ML file (43 cols)
df_ml = pd.read_csv("HR_Job_Placement_Cleaned.csv")
print("ML file shape before:", df_ml.shape)  # (50000, 43)

# Load your feature engineered file (31 cols)
df_fe = pd.read_csv("HR_Cleaned_For_ML.csv")

# Get ONLY the new feature columns
new_cols = ['experience_category', 'academic_band',
            'skills_level', 'interview_category',
            'placement_score']

# Add new columns to ML file
for col in new_cols:
    df_ml[col] = df_fe[col].values
    print(f"  {col} added ")

print("ML file shape after:", df_ml.shape)  # (50000, 48)

# Save final complete ML file
df_ml.to_csv("HR_Job_Placement_Cleaned.csv", index=False)
print(" Final ML file saved!")


ML file shape before: (50000, 43)
  experience_category added 
  academic_band added 
  skills_level added 
  interview_category added 
  placement_score added 
ML file shape after: (50000, 48)
 Final ML file saved!


In [14]:
df

,age_years,gender,ssc_percentage,hsc_percentage,degree_percentage,degree_specialization,technical_score,aptitude_score,communication_score,skills_match_percentage,...,notice_period_days,layoff_history,employment_gap_months,relocation_willingness,status,experience_category,academic_band,skills_level,interview_category,placement_score
0,27,Male,65.061656,83.842578,75.856526,Computer Science,58.221909,89.566305,64.474484,79.548913,...,15.0,No,18.0,Not Willing,Not Placed,1,1,2,2,65.87
1,24,Male,67.885626,64.973305,73.093588,Electronics,71.927978,54.591971,61.077306,73.316134,...,0.0,No,0.0,Not Willing,Not Placed,0,3,2,0,70.17
2,33,Female,73.892471,68.834121,90.196460,Information Technology,72.445041,58.587088,79.494739,75.466980,...,0.0,No,3.0,Not Willing,Placed,1,0,2,0,74.76
3,31,Male,74.145568,76.255126,75.586731,Mechanical,78.855676,61.022065,53.740386,73.676449,...,0.0,Yes,6.0,Willing,Not Placed,0,1,2,1,72.28
4,28,Male,60.475937,65.786336,80.801010,Information Technology,68.286776,65.713731,61.438314,88.994847,...,0.0,No,3.0,Willing,Not Placed,0,1,0,0,73.13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,29,Female,76.382971,82.705920,81.790211,Electronics,71.955587,74.280744,65.613587,61.997596,...,0.0,No,6.0,Willing,Not Placed,1,1,1,0,67.70
49996,32,Female,80.029965,68.803782,69.178840,Mechanical,53.488972,51.796635,75.373337,77.767081,...,30.0,No,0.0,Willing,Not Placed,0,3,2,2,65.15
49997,28,Male,61.801411,69.771590,79.777937,Mechanical,60.530457,40.000000,54.851899,54.749371,...,0.0,No,3.0,Willing,Not Placed,0,1,1,0,57.66
49998,34,Male,71.811897,76.840708,86.809844,Others,67.611868,44.373600,69.899354,92.333175,...,60.0,No,0.0,Not Willing,Not Placed,0,0,0,0,75.49


In [15]:
from sklearn.preprocessing import StandardScaler 

In [16]:
scale_cols = [
    'degree_percentage',
    'communication_score',
    'technical_score',
    'skills_match_percentage',
    'placement_score'
]

scaler = StandardScaler()                           # create the scaler tool

# fit_transform: learns the mean & std → then scales all values
df[scale_cols] = scaler.fit_transform(df[scale_cols])


In [17]:
import pandas as pd

# Load your ML file
df = pd.read_csv("HR_Job_Placement_Cleaned.csv")

# ── Check 1: Shape ──────────────────────────────────────────
print("Shape:", df.shape)
print("Total columns:", len(df.columns))

# ── Check 2: All column names with numbers ──────────────────
print("\nAll Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"  {i}. {col}")

# ── Check 3: Check new features exist ───────────────────────
print("\nNew Feature Columns Check:")
new_cols = ['experience_category', 'academic_band',
            'skills_level', 'interview_category',
            'placement_score']

for col in new_cols:
    if col in df.columns:
        print(f"  ✅ {col} EXISTS")
    else:
        print(f"  ❌ {col} MISSING")

# ── Check 4: Sample values ───────────────────────────────────
print("\nFirst 3 rows sample:")
print(df.head(3))

Shape: (50000, 48)
Total columns: 48

All Columns:
  1. age_years
  2. ssc_percentage
  3. hsc_percentage
  4. degree_percentage
  5. technical_score
  6. aptitude_score
  7. communication_score
  8. skills_match_percentage
  9. certifications_count
  10. internship_experience
  11. years_of_experience
  12. career_switch_willingness
  13. relevant_experience
  14. previous_ctc_lpa
  15. expected_ctc_lpa
  16. job_role_match
  17. bond_requirement
  18. notice_period_days
  19. layoff_history
  20. employment_gap_months
  21. relocation_willingness
  22. status
  23. status_encoded
  24. internship_experience_encoded
  25. career_switch_willingness_encoded
  26. relevant_experience_encoded
  27. job_role_match_encoded
  28. bond_requirement_encoded
  29. layoff_history_encoded
  30. relocation_willingness_encoded
  31. gender_Female
  32. gender_Male
  33. degree_specialization_Computer Science
  34. degree_specialization_Electronics
  35. degree_specialization_Information Technology
 

In [18]:
df.columns

Index(['age_years', 'ssc_percentage', 'hsc_percentage', 'degree_percentage',
       'technical_score', 'aptitude_score', 'communication_score',
       'skills_match_percentage', 'certifications_count',
       'internship_experience', 'years_of_experience',
       'career_switch_willingness', 'relevant_experience', 'previous_ctc_lpa',
       'expected_ctc_lpa', 'job_role_match', 'bond_requirement',
       'notice_period_days', 'layoff_history', 'employment_gap_months',
       'relocation_willingness', 'status', 'status_encoded',
       'internship_experience_encoded', 'career_switch_willingness_encoded',
       'relevant_experience_encoded', 'job_role_match_encoded',
       'bond_requirement_encoded', 'layoff_history_encoded',
       'relocation_willingness_encoded', 'gender_Female', 'gender_Male',
       'degree_specialization_Computer Science',
       'degree_specialization_Electronics',
       'degree_specialization_Information Technology',
       'degree_specialization_Mechanical', 

In [19]:
df.head(3)  



,age_years,ssc_percentage,hsc_percentage,degree_percentage,technical_score,aptitude_score,communication_score,skills_match_percentage,certifications_count,internship_experience,...,company_tier_Tier 2,company_tier_Tier 3,competition_level_High,competition_level_Low,competition_level_Medium,experience_category,academic_band,skills_level,interview_category,placement_score
0,-0.125697,-0.636961,1.517642,0.266272,-0.827689,2.579495,-0.164031,0.475300,0.505576,No,...,0,1,0,0,1,1,1,2,2,65.87
1,-0.870819,-0.273841,-0.913011,-0.131413,0.328398,-0.948413,-0.507189,-0.052936,-0.338204,Yes,...,0,0,1,0,0,0,3,2,0,70.17
2,1.364547,0.498548,-0.415679,2.330301,0.372012,-0.545420,1.353205,0.129351,-0.338204,Yes,...,0,1,0,1,0,1,0,2,0,74.76
